# Effect sizes — how big, not just whether

_Metrics & Evaluation — measuring models, data & statistics_

**A p-value tells you IF a difference is real; an effect size tells you HOW BIG it is and whether it matters.**

An effect size is a number that measures how big a difference or a relationship is — on a scale you can compare across studies. It is the answer to "how much?", separate from "is it real?".

       The cleanest example is Cohen's d. Two groups have means that differ. How big is that gap? Measuring it in raw units (dollars, milliseconds) is hard to compare across problems. So we measure the gap in units of the typical spread — we divide the difference of the means by the standard deviation (a measure of how spread out the values are; see the Variance lesson). A d of 1 means the groups are one whole standard deviation apart: a clearly visible separation. A d of 0.1 means they overlap almost completely.

---

This notebook is a practice scaffold for the **Effect sizes — how big, not just whether** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scipy / pingouin

In [ ]:
import numpy as np
from scipy import stats

# ---- 1) standardized mean differences (two groups of numbers) ----
def cohens_d(a, b):
    a, b = np.asarray(a), np.asarray(b)
    n1, n2 = len(a), len(b)
    # pooled standard deviation (the two groups' spreads combined)
    sp = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / sp

def hedges_g(a, b):
    # Cohen's d times a small-sample bias correction (~1 for large n)
    n = len(a) + len(b)
    return cohens_d(a, b) * (1 - 3 / (4 * n - 9))

def glass_delta(treat, control):
    # divide by the CONTROL group's spread only
    return (np.mean(treat) - np.mean(control)) / np.std(control, ddof=1)

# ---- 2) Cliff's delta (nonparametric: ordering only, robust to outliers)
def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > b).sum() for x in a)   # times an a beats a b
    lt = sum((x < b).sum() for x in a)   # times an a loses to a b
    return (gt - lt) / (len(a) * len(b))

# ---- 3) binary outcome: odds ratio & risk ratio from a 2x2 table ----
#   a = exposed+event, b = exposed+noevent, c = unexposed+event, d = unexposed+noevent
def two_by_two(a, b, c, d):
    risk_exp = a / (a + b)
    risk_unexp = c / (c + d)
    rr = risk_exp / risk_unexp                 # risk ratio (relative risk)
    rd = risk_exp - risk_unexp                 # risk difference
    nnt = 1 / rd                               # Number Needed to Treat
    odds_ratio = (a * d) / (b * c)             # odds ratio (cross-product)
    return dict(rr=rr, rd=rd, nnt=nnt, odds_ratio=odds_ratio)

# scipy also gives the odds ratio + its exact test directly:
res = stats.fisher_exact([[40, 10], [20, 30]])   # (odds_ratio, p_value)
print("scipy odds ratio:", round(res[0], 3), " p:", round(res[1], 4))

print(two_by_two(40, 10, 20, 30))
# -> rr=2.0, rd=0.40, nnt=2.5, odds_ratio=6.0

# ---- pingouin: test + effect size + benchmark label in one call ----
# import pingouin as pg
# pg.ttest(group1, group2)          # returns t, p-val, AND 'cohen-d'
# pg.compute_effsize(g1, g2, eftype='hedges')   # also 'cohen', 'glass', 'r'
# pg.anova(data=df, dv='score', between='group')  # returns p AND 'np2' (partial eta^2)

## Visualize the data & results

_On the real breast-cancer data, which features most strongly separate malignant from benign tumors — measured by Cohen's d, NOT by a p-value?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target          # target: 0 = malignant, 1 = benign
mal = X[y == 0]                        # malignant tumors
ben = X[y == 1]                        # benign tumors

def cohens_d(a, b):
    n1, n2 = len(a), len(b)
    # pooled standard deviation: the two groups' spreads combined
    sp = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1))
                 / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / sp

# Cohen's d for every feature, then rank by magnitude
ds = np.array([cohens_d(mal[:, i], ben[:, i]) for i in range(X.shape[1])])
order = np.argsort(-np.abs(ds))        # largest separation first
for i in order[:8]:
    print(f"{data.feature_names[i]:25s}  d = {ds[i]:.3f}")
# worst concave points  d = 2.693   <- biggest separation (highlighted bar)
# worst perimeter       d = 2.598
# mean concave points   d = 2.545
# ...
# symmetry error        d ~ 0.013   <- tiny separation, yet still "significant"

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate reports: "The new onboarding flow significantly increased signups (p < 0.001) across 4 million users." Why is the p-value alone not enough to decide whether to ship it, and what should you ask for?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Separate "real" from "big". — _A p-value answers only whether the difference is likely real, not how large it is._
- Account for the huge sample. — _With 4 million users, even a microscopic lift (say signups from 10.00% to 10.02%) will be "highly significant" because the test statistic grows with the square root of the sample size._
- Ask for the effect size in real units. — _Request the risk difference (percentage-point lift) and risk ratio, plus a confidence interval, so you can weigh the actual gain against the cost of shipping._

**Answer:** "p < 0.001" only says the increase is almost certainly not luck — it says nothing about how big it is, and with 4 million users essentially any non-zero effect clears that bar. Ask for the effect size: the risk difference (the absolute percentage-point lift in signup rate) and the risk ratio, with a confidence interval. If the lift is 0.02 points, it is real but probably not worth the engineering cost; if it is 3 points, ship it.

</details>

**Problem 2.** Two groups have means 78 and 72. In study A the pooled standard deviation is 12; in study B it is 3. Compute Cohen's d for each and explain why the much larger d in study B can be misleading.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply Cohen's d. — _d = (mean1 − mean2) / pooled standard deviation. The numerator (the raw gap) is the same 6 in both studies._
- Plug in each spread. — _Study A: 6 / 12 = 0.5. Study B: 6 / 3 = 2.0. Same gap, very different d, purely because the denominator changed._
- Interpret the inflation. — _A tiny within-group spread makes any gap look like many standard deviations, so d balloons even though the real-world difference (6 points) is identical._

**Answer:** Study A: $d = 6/12 = 0.5$ (medium). Study B: $d = 6/3 = 2.0$ (very large). The raw difference is the same 6 points in both — only the spread changed. Because Cohen's d divides by the pooled standard deviation, a very small within-group variance inflates d, making study B look dramatically stronger when the actual gap is identical. The fix is to always report the raw difference in real units alongside d, and to be suspicious of a giant d paired with an unusually tiny standard deviation.

</details>

**Problem 3.** A 2×2 table: of 50 exposed, 40 had the event; of 50 unexposed, 20 had the event. Compute the risk ratio, risk difference, NNT, and odds ratio. Why would reporting the odds ratio as "the exposed are 6× as likely" be wrong here?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Get the two risks. — _Risk = events / group size. Exposed: 40/50 = 0.80. Unexposed: 20/50 = 0.40._
- Form RR, RD, NNT. — _RR = 0.80/0.40 = 2.0. RD = 0.80 − 0.40 = 0.40. NNT = 1/0.40 = 2.5._
- Form the odds ratio. — _OR = (a·d)/(b·c) = (40·30)/(10·20) = 1200/200 = 6.0, using odds (event/non-event), not risk._

**Answer:** Risks are 0.80 (exposed) and 0.40 (unexposed). RR = 2.0 (twice as likely), RD = 0.40 (a 40-point absolute jump), NNT = 2.5, and OR = (40·30)/(10·20) = 6.0. Saying "6× as likely" misreads the odds ratio as a risk ratio: "how many times more likely" is the risk ratio, which is 2, not 6. Odds and risk only line up when the event is rare; here the event is common (40–80%), so the odds ratio sits much further from 1 than the risk ratio. Report RR (or RD) when you mean likelihood.

</details>